In [ ]:
!pip install -q numpy pandas scipy soundfile librosa pyloudnorm tqdm matplotlib seaborn torch torchaudio easydict


In [ ]:
import os
import re
import json
import math
import time
import shutil
import hashlib
import random
import warnings
import tempfile
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Any, Dict, Tuple, Optional

import numpy as np
import pandas as pd

import soundfile as sf
import librosa
import pyloudnorm as pyln

import torch
from tqdm.auto import tqdm

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')

RAW_DATA_ROOTS = {
    'iemocap': DRIVE_ROOT / 'Data/Raw/iemocap',
    'ravdess': DRIVE_ROOT / 'Data/Raw/ravdess',
    'crema': DRIVE_ROOT / 'Data/Raw/cremad',
}

OUT_ROOT = DRIVE_ROOT / 'data/processed_v10_safe'
AUDIO_OUT = OUT_ROOT / 'processed_audio'
FEATURE_OUT = OUT_ROOT / 'precomputed'
AUDIT_OUT = OUT_ROOT / 'audit'

for p in [OUT_ROOT, AUDIO_OUT, FEATURE_OUT, AUDIT_OUT]:
    p.mkdir(parents=True, exist_ok=True)

# Class gom toàn bộ cấu hình tiền xử lý
@dataclass
class PreprocessConfig:
    # Audio format
    sample_rate: int = 16000
    mono: bool = True
    dtype: str = 'float32'

    # Loudness
    target_lufs: float = -23.0
    peak_guard: float = 0.98
    min_lufs_valid: float = -60.0
    min_final_lufs_valid: float = -50.0
    max_gain_db: float = 35.0

    # Conservative boundary VAD
    vad_frame_ms: float = 30.0
    vad_hop_ms: float = 10.0
    vad_db_margin: float = 12.0
    vad_keep_silence_ms: float = 200.0
    vad_min_voiced_ms: float = 60.0
    max_vad_trim_ratio: float = 0.55
    vad_fallback_to_untrimmed: bool = True

    # Duration
    min_duration_sec: float = 1.2
    max_duration_sec: float = 8.0
    crop_long_strategy: str = 'center'

    # Feature extraction
    n_mels: int = 80
    n_fft: int = 512
    win_length: int = 400
    hop_length: int = 160
    f_min: float = 50.0
    f_max: float = 8000.0
    f0_min: float = 50.0
    f0_max: float = 500.0
    top_db: float = 80.0

    # Dataset selection
    use_iemocap_impro_only: bool = True
    ravdess_speech_only: bool = True
    ravdess_dedupe_by_stem: bool = True

    # Split
    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15
    split_seed: int = 42

    # Behavior
    overwrite_audio: bool = True
    overwrite_features: bool = True
    fail_fast: bool = False

    # Audit
    audit_random_feature_checks: int = 64

Các hàm nhỏ dùng lại nhiều lần:

In [ ]:
# Hàm tạo mã băm ổn định từ chuỗi đường dẫn để tránh trùng ID giữa các file âm thanh.
def stable_hash(text: str, n: int = 12) -> str:
    """Tạo hash ngắn nhưng ổn định; dùng trong sample_id để mỗi file có định danh duy nhất."""
    return hashlib.sha1(text.encode('utf-8')).hexdigest()[:n]


# Hàm làm sạch tên file để đưa vào sample_id/filename mà không chứa ký tự lạ.
def safe_filename_stem(path: Path) -> str:
    """Chuẩn hóa phần stem của tên file, chỉ giữ chữ, số, gạch dưới và gạch ngang."""
    return re.sub(r'[^A-Za-z0-9_\-]+', '_', path.stem)


# Hàm tạo sample_id duy nhất cho từng file âm thanh dựa trên dataset, đường dẫn và tên file.
def make_sample_id(dataset: str, source_path: Path) -> str:
    """Ghép dataset + hash đường dẫn + tên file đã làm sạch để tạo khóa chính cho metadata."""
    stem = safe_filename_stem(source_path)
    h = stable_hash(str(source_path), 12)
    return f'{dataset}__{h}__{stem}'


# Hàm tạo feature_id tương ứng với sample_id; hiện giữ nguyên để một-mẫu-một-feature.
def make_feature_id(sample_id: str) -> str:
    """Trả về định danh feature; tách riêng để sau này dễ đổi quy tắc đặt tên feature."""
    return sample_id


# Hàm lưu file .npy theo kiểu an toàn: ghi vào file tạm trước rồi mới thay thế file đích.
def atomic_numpy_save(path: Path, arr: np.ndarray):
    """Giảm nguy cơ hỏng file nếu Colab/Drive bị ngắt giữa lúc đang ghi processed audio."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(delete=False, dir=str(path.parent), suffix='.npy') as tmp:
        tmp_path = Path(tmp.name)
    try:
        np.save(tmp_path, arr)
        os.replace(tmp_path, path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)


# Hàm lưu file .pt theo kiểu an toàn: ghi tạm rồi đổi tên sang file đích.
def atomic_torch_save(path: Path, obj: Dict[str, Any]):
    """Dùng để lưu precomputed features; tránh tạo file feature bị ghi dở."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(delete=False, dir=str(path.parent), suffix='.pt') as tmp:
        tmp_path = Path(tmp.name)
    try:
        torch.save(obj, tmp_path)
        os.replace(tmp_path, path)
    finally:
        if tmp_path.exists():
            tmp_path.unlink(missing_ok=True)


# Hàm chuẩn hóa z-score an toàn cho chuỗi đặc trưng, có xử lý NaN/Inf và độ lệch chuẩn quá nhỏ.
def safe_zscore_np(x: np.ndarray, mask: Optional[np.ndarray] = None, eps: float = 1e-8) -> np.ndarray:
    """Chuẩn hóa mảng về thang tương đối ổn định; nếu có mask thì chỉ dùng vùng mask để tính mean/std."""
    # Ép kiểu float32 để thống nhất kiểu dữ liệu giữa numpy và torch.
    x = np.asarray(x, dtype=np.float32)
    # Thay NaN/Inf bằng 0 để lỗi dữ liệu không lan sang training.
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    # Nếu có mask, chỉ thống kê trên vùng hợp lệ, ví dụ vùng voiced của F0.
    if mask is not None:
        mask = np.asarray(mask).astype(bool)
        if mask.sum() > 1:
            mu = float(x[mask].mean())
            sd = float(x[mask].std())
        # Nhánh B: chế độ trình bày — không chạy lại xử lý dữ liệu, chỉ load metadata đã có.
        else:
            mu = float(x.mean()) if x.size else 0.0
            sd = float(x.std()) if x.size else 1.0
    # Nhánh B: chế độ trình bày — không chạy lại xử lý dữ liệu, chỉ load metadata đã có.
    else:
        mu = float(x.mean()) if x.size else 0.0
        sd = float(x.std()) if x.size else 1.0

    # Nếu độ lệch chuẩn quá nhỏ, đặt sd=1 để tránh chia cho 0.
    if not np.isfinite(sd) or sd < eps:
        sd = 1.0
    return ((x - mu) / (sd + eps)).astype(np.float32)


# Hàm này căn chiều dài đặc trưng 1D về đúng số frame T của mel-spectrogram.
def align_1d_to_T(x: np.ndarray, T: int) -> np.ndarray:
    """Nếu đặc trưng dài hơn thì cắt, ngắn hơn thì pad bằng giá trị cuối để đồng bộ với mel."""
    x = np.asarray(x, dtype=np.float32).reshape(-1)
    if len(x) == T:
        return x
    if len(x) > T:
        return x[:T]
    if len(x) == 0:
        return np.zeros(T, dtype=np.float32)
    pad = np.full(T - len(x), x[-1], dtype=np.float32)
    return np.concatenate([x, pad], axis=0).astype(np.float32)


# Hàm này kiểm tra đặc trưng không chứa NaN hoặc Inf trước khi lưu vào feature file.
def assert_finite_np(name: str, x: np.ndarray):
    """Dừng pipeline sớm nếu feature có giá trị không hợp lệ, tránh lỗi âm thầm ở notebook training."""
    if not np.isfinite(x).all():
        raise ValueError(f'Non-finite values detected in {name}')

In [ ]:
FINAL_EMOTIONS = ['anger', 'fear', 'happiness', 'neutral', 'sadness']
EMOTION_TO_ID = {e: i for i, e in enumerate(FINAL_EMOTIONS)}

with open(OUT_ROOT / 'emotion_map.json', 'w', encoding='utf-8') as f:
    json.dump({
        'final_emotions': FINAL_EMOTIONS,
        'emotion_to_id': EMOTION_TO_ID,
        'id_to_emotion': {i: e for e, i in EMOTION_TO_ID.items()},
    }, f, indent=2, ensure_ascii=False)

# Hàm này chuẩn hóa nhãn cảm xúc gốc của từng dataset về 5 nhãn dùng chung trong đề tài.
def normalize_emotion(dataset: str, raw: str) -> Optional[str]:
    """Map nhãn thô của từng corpus về anger/fear/happiness/neutral/sadness và loại nhãn ngoài phạm vi."""
    # Chuẩn hóa chuỗi nhãn thô để so khớp không phụ thuộc viết hoa/thường.
    r = str(raw).strip().lower()

    # Map nhãn IEMOCAP: giữ 5 lớp chính; loại các nhãn ngoài phạm vi đề tài.
    iemocap_map = {
        'ang': 'anger',
        'hap': 'happiness',
        'exc': 'happiness',
        'neu': 'neutral',
        'sad': 'sadness',
        'fea': 'fear',
        'fru': None,
        'sur': None,
        'dis': None,
        'oth': None,
        'xxx': None,
    }

    # Map nhãn RAVDESS: giữ 5 lớp chính; loại calm/disgust/surprise để đồng bộ corpus.
    ravdess_map = {
        '01': 'neutral',
        '02': None,
        '03': 'happiness',
        '04': 'sadness',
        '05': 'anger',
        '06': 'fear',
        '07': None,
        '08': None,
    }
    # Map nhãn CREMA-D: giữ 5 lớp chính; loại disgust khỏi phạm vi.
    crema_map = {
        'ANG': 'anger',
        'FEA': 'fear',
        'HAP': 'happiness',
        'NEU': 'neutral',
        'SAD': 'sadness',
        'DIS': None,
    }

    if dataset == 'iemocap':
        return iemocap_map.get(r, None)
    if dataset == 'ravdess':
        return ravdess_map.get(r, None)
    if dataset == 'crema':
        return crema_map.get(str(raw).upper(), None)
    return None


# Hàm này đọc metadata IEMOCAP: lấy nhãn từ EmoEvaluation, ghép với wav, lọc nhãn hợp lệ và tạo speaker_uid.
def parse_iemocap(root: Path) -> pd.DataFrame:
    """Trả về DataFrame metadata cho IEMOCAP; có thể lọc improvised utterances theo CFG."""
    records = []
    if not root.exists():
        print('IEMOCAP root not found:', root)
        return pd.DataFrame()

    # Bước 1: tạo bảng tra utterance_id → raw emotion từ file EmoEvaluation của IEMOCAP.
    label_map = {}
    eval_files = sorted(root.glob('Session*/dialog/EmoEvaluation/*.txt'))
    pat = re.compile(r'^\[(.*?)\]\s+(\S+)\s+(\S+)\s+\[.*?\]')

    for ef in eval_files:
        try:
            lines = ef.read_text(errors='ignore').splitlines()
        except Exception:
            continue
        for line in lines:
            m = pat.match(line.strip())
            if m:
                utt_id = m.group(2)
                emo = m.group(3)
                label_map[utt_id] = emo

    # Bước 2: tìm toàn bộ file wav dạng sentence-level của IEMOCAP.
    wavs = sorted(root.glob('Session*/sentences/wav/**/*.wav'))

    # Bước 3: duyệt từng wav, lọc improvised nếu cần, rồi ghép với nhãn.
    for wav in tqdm(wavs, desc='Parse IEMOCAP'):
        utt_id = wav.stem
        if CFG.use_iemocap_impro_only and 'impro' not in utt_id.lower():
            continue

        raw_emo = label_map.get(utt_id, None)
        if raw_emo is None:
            continue

        emo = normalize_emotion('iemocap', raw_emo)
        if emo is None:
            continue

        session_match = re.search(r'(Session\d+)', str(wav))
        session = session_match.group(1) if session_match else 'unknown_session'

        spk_prefix = utt_id.split('_')[0] if '_' in utt_id else utt_id[:6]
        speaker_uid = f'iemocap__{spk_prefix}'

        records.append({
            'dataset': 'iemocap',
            'source_path': str(wav),
            'relative_path': str(wav.relative_to(root)) if root in wav.parents else str(wav.name),
            'filename': wav.name,
            'utterance_id': utt_id,
            'speaker_local_id': spk_prefix,
            'speaker_uid': speaker_uid,
            'emotion': emo,
            'emotion_label': EMOTION_TO_ID[emo],
            'raw_emotion': raw_emo,
            'session': session,
            'is_improvised': 'impro' in utt_id.lower(),
        })

    return pd.DataFrame(records)


# Hàm này đọc metadata RAVDESS từ quy ước tên file, lọc speech-only, khử trùng lặp stem và tạo speaker_uid.
def parse_ravdess(root: Path) -> pd.DataFrame:
    """RAVDESS mã hóa emotion/speaker trong tên file; parser tách các trường đó thành metadata."""
    records = []
    if not root.exists():
        print('RAVDESS root not found:', root)
        return pd.DataFrame()

    # Tìm tất cả wav trong RAVDESS, kể cả khi thư mục có nhiều bản copy/nhánh Actor_*.
    all_wavs = sorted(root.glob('**/*.wav'))

    # Khử trùng lặp stem RAVDESS để tránh nhân đôi cùng một utterance.
    if CFG.ravdess_dedupe_by_stem:
        stem_to_paths = {}
        for wav in all_wavs:
            stem_to_paths.setdefault(wav.stem, []).append(wav)

        selected_wavs = []
        duplicate_groups = []

        for stem, paths in stem_to_paths.items():
            if len(paths) > 1:
                duplicate_groups.append((stem, [str(p) for p in paths]))

            paths = sorted(
                paths,
                key=lambda p: (
                    0 if any(part.startswith('Actor_') for part in p.parts) else 1,
                    len(str(p))
                )
            )
            selected_wavs.append(paths[0])

        selected_wavs = sorted(selected_wavs)

        if duplicate_groups:
            dup_path = AUDIT_OUT / 'ravdess_duplicate_stems_before_selection.json'
            with open(dup_path, 'w', encoding='utf-8') as f:
                json.dump(
                    [{'stem': s, 'paths': ps} for s, ps in duplicate_groups],
                    f,
                    indent=2,
                    ensure_ascii=False,
                )
            print('RAVDESS duplicate stem groups:', len(duplicate_groups))
            print('Saved duplicate stem report:', dup_path)
    # Nhánh B: chế độ trình bày — không chạy lại xử lý dữ liệu, chỉ load metadata đã có.
    else:
        selected_wavs = all_wavs

    print(f'RAVDESS wav files found: {len(all_wavs)}')
    print(f'RAVDESS selected unique stems: {len(selected_wavs)}')

    # Duyệt các wav đã chọn và tách metadata trực tiếp từ tên file RAVDESS.
    for wav in tqdm(selected_wavs, desc='Parse RAVDESS'):
        stem = wav.stem
        parts = stem.split('-')
        if len(parts) != 7:
            continue

        modality, vocal_channel, emotion_code, intensity, statement, repetition, actor = parts

        # RAVDESS vocal_channel 01 = speech, 02 = song.
        # Chỉ giữ speech, loại song để dữ liệu đồng nhất hơn với IEMOCAP/CREMA-D.
        if CFG.ravdess_speech_only and vocal_channel != '01':
            continue

        emo = normalize_emotion('ravdess', emotion_code)
        if emo is None:
            continue

        speaker_uid = f'ravdess__actor_{actor}'

        records.append({
            'dataset': 'ravdess',
            'source_path': str(wav),
            'relative_path': str(wav.relative_to(root)) if root in wav.parents else str(wav.name),
            'filename': wav.name,
            'utterance_id': stem,
            'speaker_local_id': actor,
            'speaker_uid': speaker_uid,
            'emotion': emo,
            'emotion_label': EMOTION_TO_ID[emo],
            'raw_emotion': emotion_code,
            'ravdess_modality': modality,
            'ravdess_vocal_channel': vocal_channel,
            'ravdess_intensity': intensity,
            'ravdess_statement': statement,
            'ravdess_repetition': repetition,
        })

    out = pd.DataFrame(records)
    if len(out):
        dup = out['utterance_id'].duplicated().sum()
        if dup:
            raise RuntimeError(f'RAVDESS parser still has duplicate utterance_id: {dup}')
    return out


# Hàm này đọc metadata CREMA-D từ tên file, chuẩn hóa emotion và tạo speaker_uid.
def parse_crema(root: Path) -> pd.DataFrame:
    """CREMA-D dùng tên file dạng speaker_sentence_emotion_level; parser tách và lưu các thành phần cần thiết."""
    records = []
    if not root.exists():
        print('CREMA-D root not found:', root)
        return pd.DataFrame()

    # CREMA-D thường lưu wav trực tiếp trong root; tên file đã chứa speaker/emotion.
    wavs = sorted(root.glob('*.wav'))

    # Duyệt từng file CREMA-D và tách speaker/sentence/emotion/level từ tên file.
    for wav in tqdm(wavs, desc='Parse CREMA-D'):
        stem = wav.stem
        parts = stem.split('_')
        if len(parts) < 4:
            continue

        speaker, sentence, emo_code, level = parts[:4]
        emo = normalize_emotion('crema', emo_code)
        if emo is None:
            continue

        speaker_uid = f'crema__speaker_{speaker}'

        records.append({
            'dataset': 'crema',
            'source_path': str(wav),
            'relative_path': str(wav.relative_to(root)) if root in wav.parents else str(wav.name),
            'filename': wav.name,
            'utterance_id': stem,
            'speaker_local_id': speaker,
            'speaker_uid': speaker_uid,
            'emotion': emo,
            'emotion_label': EMOTION_TO_ID[emo],
            'raw_emotion': emo_code,
            'crema_sentence': sentence,
            'crema_level': level,
        })

    return pd.DataFrame(records)


# Hàm này gom metadata của IEMOCAP, RAVDESS và CREMA-D thành một bảng thống nhất.
def parse_all_datasets() -> pd.DataFrame:
    """Sau khi parse từng corpus, hàm tạo sample_id, feature_id, đường dẫn processed/feature và kiểm tra trùng lặp."""
    # Gọi parser của từng corpus rồi gom lại thành một DataFrame chung.
    dfs = [
        parse_iemocap(RAW_DATA_ROOTS['iemocap']),
        parse_ravdess(RAW_DATA_ROOTS['ravdess']),
        parse_crema(RAW_DATA_ROOTS['crema']),
    ]
    dfs = [d for d in dfs if len(d) > 0]
    if not dfs:
        raise RuntimeError('No datasets parsed. Check RAW_DATA_ROOTS.')

    df = pd.concat(dfs, ignore_index=True)

    sample_ids = []
    feature_ids = []
    processed_relpaths = []
    feature_relpaths = []

    # Tạo ID và đường dẫn output cho từng sample; chưa xử lý audio ở bước này.
    for _, row in df.iterrows():
        # Lấy đường dẫn raw audio từ metadata.
        src = Path(row['source_path'])
        sid = make_sample_id(row['dataset'], src)
        fid = make_feature_id(sid)

        sample_ids.append(sid)
        feature_ids.append(fid)
        processed_relpaths.append(f'{row["dataset"]}/{fid}.npy')
        feature_relpaths.append(f'{fid}.pt')

    df['sample_id'] = sample_ids
    df['feature_id'] = feature_ids
    df['processed_relpath'] = processed_relpaths
    df['feature_relpath'] = feature_relpaths

    # Kiểm tra và loại trùng sample_id để mỗi mẫu chỉ xuất hiện một lần.
    before = len(df)
    df = df.drop_duplicates('sample_id').reset_index(drop=True)
    after = len(df)
    if before != after:
        print(f'Dropped duplicate sample_id rows: {before - after}')

    # Extra corpus-level duplicate protection.
    if 'ravdess' in set(df['dataset']):
        # Nhóm 6: kiểm tra riêng RAVDESS vì dễ bị duplicate nếu folder có bản copy.
        rav = df[df['dataset'] == 'ravdess']
        dup_rav = rav['utterance_id'].duplicated().sum()
        if dup_rav:
            raise RuntimeError(f'Duplicate RAVDESS utterance_id after parse: {dup_rav}')

    return df

Tiền xử lý âm thanh

In [ ]:
# Hàm đọc âm thanh, chuyển mono, xử lý NaN/Inf, resample về 16 kHz và đưa biên độ về miền an toàn.
def load_audio_mono(path: Path, sr: int) -> Tuple[np.ndarray, int]:
    """Bước chuẩn hóa waveform đầu tiên trước LUFS, VAD và trích xuất đặc trưng."""
    # Đọc âm thanh bằng soundfile; always_2d=False giữ nguyên mono nếu file vốn là mono.
    wav, native_sr = sf.read(str(path), always_2d=False)

    # Nếu stereo/multi-channel, lấy trung bình các kênh để đưa về mono.
    if wav.ndim > 1:
        wav = wav.mean(axis=1)

    wav = wav.astype(np.float32)
    wav = np.nan_to_num(wav, nan=0.0, posinf=0.0, neginf=0.0)

    # Resample về sample_rate thống nhất để feature có cùng thang thời gian.
    if native_sr != sr:
        wav = librosa.resample(wav, orig_sr=native_sr, target_sr=sr).astype(np.float32)

    # Đo peak để phát hiện biên độ bất thường và/hoặc tránh clipping.
    peak = float(np.max(np.abs(wav))) if len(wav) else 0.0
    if peak > 1.5:
        wav = wav / (peak + 1e-8)

    return wav.astype(np.float32), sr


# Hàm đo loudness theo LUFS để kiểm soát độ lớn cảm nhận giữa các file và dataset.
def measure_lufs(wav: np.ndarray, sr: int) -> float:
    """Trả về -inf với file quá ngắn/quá nhỏ/gần im lặng để pipeline có thể loại hoặc xử lý an toàn."""
    # Chuẩn hóa waveform đầu vào trước khi trích xuất feature.
    wav = np.asarray(wav, dtype=np.float32)
    wav = np.nan_to_num(wav, nan=0.0, posinf=0.0, neginf=0.0)

    if len(wav) < int(sr * 0.4):
        return float('-inf')

    # Đo peak để phát hiện biên độ bất thường và/hoặc tránh clipping.
    peak = float(np.max(np.abs(wav))) if len(wav) else 0.0
    rms = float(np.sqrt(np.mean(wav ** 2) + 1e-12)) if len(wav) else 0.0

    if peak < 1e-5 or rms < 1e-6:
        return float('-inf')

    meter = pyln.Meter(sr)
    try:
        loudness = meter.integrated_loudness(wav.astype(np.float64))
    except Exception:
        loudness = float('-inf')

    if not np.isfinite(loudness):
        return float('-inf')

    return float(loudness)


# Hàm giới hạn peak tối đa để tránh clipping sau khi tăng/giảm loudness.
def peak_normalize_guard(wav: np.ndarray, peak_guard: float) -> np.ndarray:
    """Nếu biên độ lớn hơn peak_guard thì scale toàn bộ waveform xuống mức an toàn."""
    # Đo peak để phát hiện biên độ bất thường và/hoặc tránh clipping.
    peak = float(np.max(np.abs(wav))) if len(wav) else 0.0
    if peak > peak_guard:
        wav = wav * (peak_guard / (peak + 1e-8))
    return wav.astype(np.float32)


# Hàm chuẩn hóa loudness về target LUFS nhưng giới hạn gain để không khuếch đại quá mạnh file gần im lặng.
def normalize_lufs(wav: np.ndarray, sr: int) -> Tuple[np.ndarray, float, float, float]:
    """Trả về waveform đã chuẩn hóa, LUFS trước/sau và gain_db để ghi vào metadata audit."""
    # Đo loudness ban đầu để biết file cần tăng/giảm bao nhiêu dB.
    lufs_before = measure_lufs(wav, sr)

    # Với file gần im lặng hoặc LUFS lỗi, không khuếch đại mạnh để tránh tạo nhiễu giả.
    if not np.isfinite(lufs_before) or lufs_before < CFG.min_lufs_valid:
        # Do not amplify near-silent files. They will be rejected by final LUFS check.
        wav_out = peak_normalize_guard(wav, CFG.peak_guard)
        lufs_after = measure_lufs(wav_out, sr)
        return wav_out.astype(np.float32), float(lufs_before), float(lufs_after), 0.0

    # Tính gain cần thiết để đưa loudness về target_lufs.
    gain_db = CFG.target_lufs - lufs_before
    # Giới hạn gain để không tăng/giảm âm lượng quá cực đoan.
    gain_db = float(np.clip(gain_db, -CFG.max_gain_db, CFG.max_gain_db))
    gain = 10.0 ** (gain_db / 20.0)

    wav_out = wav * gain
    wav_out = peak_normalize_guard(wav_out, CFG.peak_guard)
    lufs_after = measure_lufs(wav_out, sr)

    return wav_out.astype(np.float32), float(lufs_before), float(lufs_after), gain_db


# Hàm tính RMS theo frame và đổi sang dB, dùng làm tín hiệu nền cho VAD biên.
def frame_rms_db(wav: np.ndarray, frame_len: int, hop_len: int) -> np.ndarray:
    """Không dùng để kết luận cảm xúc; chỉ hỗ trợ tìm vùng có tiếng nói để trim an toàn."""
    if len(wav) < frame_len:
        wav = np.pad(wav, (0, frame_len - len(wav)))
    frames = librosa.util.frame(wav, frame_length=frame_len, hop_length=hop_len)
    rms = np.sqrt(np.mean(frames ** 2, axis=0) + 1e-12)
    db = 20.0 * np.log10(rms + 1e-12)
    return db.astype(np.float32)


# Hàm làm mượt mask True/False để tránh VAD bị đứt đoạn vì các frame lẻ.
def smooth_bool(mask: np.ndarray, min_true_frames: int) -> np.ndarray:
    """Dùng convolution nhỏ để nối các vùng voiced gần nhau, giúp điểm cắt bớt nhiễu."""
    mask = np.asarray(mask).astype(bool)
    if len(mask) == 0:
        return mask
    kernel = np.ones(max(1, min_true_frames), dtype=np.float32)
    conv = np.convolve(mask.astype(np.float32), kernel, mode='same')
    return conv > 0


# Hàm cắt khoảng lặng đầu/cuối bằng VAD bảo thủ, có fallback nếu việc cắt làm mất quá nhiều âm thanh.
def boundary_vad_trim_safe(wav: np.ndarray, sr: int) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Mục tiêu là giảm silence nhưng không cắt hỏng câu nói; mọi quyết định được ghi vào vad_info để audit."""
    original_len = len(wav)
    original_duration = original_len / sr

    # Chuyển cấu hình VAD từ mili-giây sang số sample/frame.
    frame_len = int(sr * CFG.vad_frame_ms / 1000.0)
    hop_len = int(sr * CFG.vad_hop_ms / 1000.0)

    # Tính năng lượng theo frame để xác định vùng có tiếng nói ở đầu/cuối.
    db = frame_rms_db(wav, frame_len, hop_len)

    # Ước lượng noise floor/median/peak để đặt ngưỡng VAD thích nghi theo từng file.
    noise_floor = float(np.percentile(db, 10))
    median_level = float(np.percentile(db, 50))
    peak_level = float(np.percentile(db, 95))

    # Ngưỡng VAD bảo thủ: tránh cắt nhầm những đoạn nói nhỏ.
    threshold = min(noise_floor + CFG.vad_db_margin, median_level - 3.0)
    threshold = min(threshold, peak_level - 25.0)

    # Tạo mask voiced theo ngưỡng năng lượng.
    voiced = db > threshold
    min_voiced_frames = max(1, int(CFG.vad_min_voiced_ms / CFG.vad_hop_ms))
    voiced = smooth_bool(voiced, min_voiced_frames)

    info = {
        'vad_threshold_db': float(threshold),
        'vad_noise_floor_db': float(noise_floor),
        'vad_median_db': float(median_level),
        'vad_peak_db': float(peak_level),
        'vad_voiced_ratio': float(voiced.mean()) if len(voiced) else 0.0,
        'vad_fallback_used': False,
        'vad_trim_ratio': 0.0,
        'vad_start_sample': 0,
        'vad_end_sample': int(original_len),
        'vad_reason': 'ok',
    }

    # Nếu không tìm thấy vùng voiced, giữ nguyên waveform thay vì cắt rỗng.
    if voiced.sum() == 0:
        info['vad_fallback_used'] = True
        info['vad_reason'] = 'no_voiced_frame'
        return wav.astype(np.float32), info

    # Tìm frame voiced đầu tiên/cuối cùng để xác định biên cần giữ.
    first = int(np.argmax(voiced))
    last = int(len(voiced) - 1 - np.argmax(voiced[::-1]))

    # Giữ lại một ít silence ở hai đầu để không cắt cụt âm đầu/cuối câu.
    keep_frames = int(CFG.vad_keep_silence_ms / CFG.vad_hop_ms)
    first = max(0, first - keep_frames)
    last = min(len(voiced) - 1, last + keep_frames)

    start_sample = max(0, first * hop_len)
    end_sample = min(original_len, last * hop_len + frame_len)

    # Cắt waveform theo biên VAD đã tính.
    trimmed = wav[start_sample:end_sample]
    trim_ratio = 1.0 - (len(trimmed) / max(original_len, 1))
    trimmed_duration = len(trimmed) / sr

    info['vad_trim_ratio'] = float(trim_ratio)
    info['vad_start_sample'] = int(start_sample)
    info['vad_end_sample'] = int(end_sample)

    # Nếu trim xong quá ngắn, fallback về bản chưa trim để tránh mất dữ liệu có ích.
    if CFG.vad_fallback_to_untrimmed and original_duration >= CFG.min_duration_sec and trimmed_duration < CFG.min_duration_sec:
        info['vad_fallback_used'] = True
        info['vad_reason'] = 'trimmed_below_min_duration'
        return wav.astype(np.float32), info

    # Nếu VAD cắt quá nhiều so với bản gốc, xem là rủi ro và fallback.
    if CFG.vad_fallback_to_untrimmed and trim_ratio > CFG.max_vad_trim_ratio:
        info['vad_fallback_used'] = True
        info['vad_reason'] = 'trim_ratio_too_high'
        return wav.astype(np.float32), info

    return trimmed.astype(np.float32), info


# Hàm kiểm soát độ dài mẫu sau VAD: loại mẫu quá ngắn và crop giữa mẫu quá dài.
def duration_control(wav: np.ndarray, sr: int) -> Tuple[np.ndarray, Dict[str, Any]]:
    """Giúp dữ liệu huấn luyện có độ dài ổn định, giảm chi phí và giảm shortcut theo duration."""
    # Tính duration hiện tại để kiểm tra min/max duration.
    duration = len(wav) / sr

    info = {
        'duration_after_vad_or_fallback': float(duration),
        'duration_final': float(duration),
        'too_short': False,
        'truncated': False,
        'crop_start_sample': 0,
        'crop_end_sample': int(len(wav)),
    }

    # Loại mẫu quá ngắn vì không đủ ngữ cảnh để học dynamics/prosody.
    if duration < CFG.min_duration_sec:
        info['too_short'] = True
        raise ValueError(f'Too short after safe VAD/fallback: {duration:.3f}s')

    max_samples = int(CFG.max_duration_sec * sr)

    # Crop mẫu quá dài để batch training ổn định và giảm shortcut theo duration.
    if duration > CFG.max_duration_sec:
        if CFG.crop_long_strategy == 'center':
            center = len(wav) // 2
            start = max(0, center - max_samples // 2)
            end = start + max_samples
            wav = wav[start:end]

            info['truncated'] = True
            info['crop_start_sample'] = int(start)
            info['crop_end_sample'] = int(end)
            info['duration_final'] = float(len(wav) / sr)
        elif CFG.crop_long_strategy == 'none':
            pass
        # Nhánh B: chế độ trình bày — không chạy lại xử lý dữ liệu, chỉ load metadata đã có.
        else:
            raise ValueError(f'Unknown crop_long_strategy: {CFG.crop_long_strategy}')

    return wav.astype(np.float32), info


# Hàm xử lý hoàn chỉnh một file âm thanh: load → LUFS → VAD → duration control → lưu processed audio.
def process_one_audio(row: pd.Series) -> Dict[str, Any]:
    """Trả về metadata chi tiết cho từng file, gồm duration, LUFS, peak, trạng thái VAD và đường dẫn output."""
    # Lấy đường dẫn raw audio từ metadata.
    src = Path(row['source_path'])
    if not src.exists():
        raise FileNotFoundError(str(src))

    out_path = AUDIO_OUT / row['processed_relpath']

    # Bước 1: load mono + resample + biên độ an toàn.
    wav, sr = load_audio_mono(src, CFG.sample_rate)
    original_duration = len(wav) / sr

    if original_duration < CFG.min_duration_sec:
        raise ValueError(f'Too short before VAD: {original_duration:.3f}s')

    # Bước 2: chuẩn hóa loudness để giảm shortcut do âm lượng/thiết bị thu.
    wav_lufs, lufs_before, lufs_after_norm, gain_db = normalize_lufs(wav, sr)

    # Bước 3: trim silence ở biên bằng VAD bảo thủ.
    wav_trim, vad_info = boundary_vad_trim_safe(wav_lufs, sr)
    # Bước 4: kiểm soát duration cuối cùng trước khi lưu.
    wav_final, dur_info = duration_control(wav_trim, sr)
    wav_final = peak_normalize_guard(wav_final, CFG.peak_guard)

    # Bước 5: kiểm tra LUFS cuối; file lỗi sẽ bị loại để không đưa vào training.
    lufs_final = measure_lufs(wav_final, sr)
    if (not np.isfinite(lufs_final)) or (lufs_final < CFG.min_final_lufs_valid):
        raise ValueError(f'Invalid final LUFS: {lufs_final}')

    # Bước 6: lưu waveform đã xử lý dưới dạng .npy để training load nhanh hơn wav gốc.
    atomic_numpy_save(out_path, wav_final.astype(np.float32))

    info = {
        'processed_path': str(out_path),
        'original_duration': float(original_duration),
        'sample_rate': int(sr),
        'lufs_before': float(lufs_before),
        'lufs_after_norm': float(lufs_after_norm),
        'lufs_final': float(lufs_final),
        'lufs_gain_db': float(gain_db),
        'peak_final': float(np.max(np.abs(wav_final))) if len(wav_final) else 0.0,
        'status': 'ok',
    }
    info.update(vad_info)
    info.update(dur_info)
    return info


# Hàm chạy process_one_audio cho toàn bộ metadata và tách rõ mẫu xử lý thành công/thất bại.
def process_all_audio(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Lưu processed_metadata_ok.csv và processed_metadata_failed.csv để kiểm tra chất lượng pipeline."""
    records = []
    errors = []

    # Chạy xử lý từng file; lỗi một file không làm dừng toàn pipeline nếu fail_fast=False.
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Process audio'):
        base = row.to_dict()
        try:
            info = process_one_audio(row)
            base.update(info)
            records.append(base)
        except Exception as e:
            base['status'] = 'failed'
            base['error'] = repr(e)
            errors.append(base)
            if CFG.fail_fast:
                raise

    ok = pd.DataFrame(records)
    err = pd.DataFrame(errors)

    ok.to_csv(OUT_ROOT / 'processed_metadata_ok.csv', index=False)
    err.to_csv(OUT_ROOT / 'processed_metadata_failed.csv', index=False)

    print('Processed OK:', len(ok))
    print('Failed:', len(err))
    if len(err):
        print(err['error'].value_counts().head(25))

    return ok.reset_index(drop=True), err.reset_index(drop=True)

Cell này đảm bảo cùng một `speaker_uid` không xuất hiện đồng thời ở train, validation và test.

In [ ]:
# Hàm chia train/validation/test theo speaker_uid, đảm bảo cùng một người nói không xuất hiện ở nhiều split.
def speaker_disjoint_split(df: pd.DataFrame) -> pd.DataFrame:
    """Đây là bước chống speaker leakage, quan trọng để kết quả không bị học thuộc người nói."""
    rng = random.Random(CFG.split_seed)
    df = df.copy().reset_index(drop=True)
    df['split'] = None

    for dataset, ddf in df.groupby('dataset'):
        speakers = []
        for spk, sdf in ddf.groupby('speaker_uid'):
            speakers.append((spk, len(sdf)))

        rng.shuffle(speakers)
        speakers = sorted(speakers, key=lambda x: x[1], reverse=True)

        n_total = len(ddf)
        targets = {
            'train': CFG.train_ratio * n_total,
            'val': CFG.val_ratio * n_total,
            'test': CFG.test_ratio * n_total,
        }
        counts = {'train': 0, 'val': 0, 'test': 0}
        speaker_to_split = {}

        for spk, n in speakers:
            deficits = {k: (targets[k] - counts[k]) / max(targets[k], 1.0) for k in counts}
            chosen = max(deficits, key=deficits.get)
            speaker_to_split[spk] = chosen
            counts[chosen] += n

        idx = df['dataset'].eq(dataset)
        df.loc[idx, 'split'] = df.loc[idx, 'speaker_uid'].map(speaker_to_split)

    if df['split'].isna().any():
        raise RuntimeError('Some rows were not assigned split')

    # Nhóm 5: kiểm tra speaker-disjoint; đây là điều kiện quan trọng nhất để chống speaker leakage.
    for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
        A = set(df[df['split'] == a]['speaker_uid'])
        B = set(df[df['split'] == b]['speaker_uid'])
        overlap = A & B
        if overlap:
            raise RuntimeError(f'Speaker leakage {a}/{b}: {len(overlap)}')

    tab = pd.crosstab(df['split'], df['emotion'])
    if (tab == 0).any().any():
        print('Warning: some split has zero samples for some emotion:')
        print(tab)

    return df.reset_index(drop=True)

In [ ]:
Trích xuất đặc trưng

In [ ]:
# Hàm trích xuất feature từ waveform đã xử lý: mel, F0, energy, ZCR/speaking activity và voiced mask.
def extract_features_from_waveform(wav: np.ndarray, sr: int) -> Dict[str, Any]:
    """Đầu ra là dict tensor dùng trực tiếp cho notebook Pha 1/Pha 2 và tạo nhãn đại diện A/T/S."""
    # Chuẩn hóa waveform đầu vào trước khi trích xuất feature.
    wav = np.asarray(wav, dtype=np.float32)
    wav = np.nan_to_num(wav, nan=0.0, posinf=0.0, neginf=0.0)

    # Nhánh mel: tạo mel-spectrogram power, biểu diễn phổ-thời gian chính.
    mel_power = librosa.feature.melspectrogram(
        y=wav,
        sr=sr,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        win_length=CFG.win_length,
        n_mels=CFG.n_mels,
        fmin=CFG.f_min,
        fmax=CFG.f_max,
        power=2.0,
    )

    # mel tương đối: chuẩn hóa theo đỉnh của chính mẫu, phù hợp cho backbone học cấu trúc phổ.
    mel_db_rel = librosa.power_to_db(mel_power, ref=np.max, top_db=CFG.top_db).astype(np.float32)
    mel = safe_zscore_np(mel_db_rel)

    # mel_abs_db giữ mức dB tuyệt đối hơn, phục vụ audit/diễn giải bổ sung.
    mel_abs_db = librosa.power_to_db(mel_power, ref=1.0, top_db=CFG.top_db).astype(np.float32)
    mel_abs = safe_zscore_np(mel_abs_db)

    # T là số frame theo thời gian; mọi đặc trưng 1D sẽ được căn về T.
    T = mel.shape[1]

    # Energy/RMS: đặc trưng mức mạnh-yếu theo frame.
    rms = librosa.feature.rms(
        y=wav,
        frame_length=CFG.win_length,
        hop_length=CFG.hop_length,
        center=True,
    )[0].astype(np.float32)
    rms = align_1d_to_T(rms, T)

    # ZCR/speaking activity proxy: chỉ báo biến thiên/hoạt động phát âm đơn giản.
    zcr = librosa.feature.zero_crossing_rate(
        y=wav,
        frame_length=CFG.win_length,
        hop_length=CFG.hop_length,
        center=True,
    )[0].astype(np.float32)
    zcr = align_1d_to_T(zcr, T)

    try:
        f0 = librosa.yin(
            wav.astype(np.float32),
            fmin=CFG.f0_min,
            fmax=CFG.f0_max,
            sr=sr,
            frame_length=CFG.win_length,
            hop_length=CFG.hop_length,
        ).astype(np.float32)
    except Exception:
        f0 = np.zeros(T, dtype=np.float32)

    f0 = align_1d_to_T(f0, T)
    f0 = np.nan_to_num(f0, nan=0.0, posinf=0.0, neginf=0.0)

    # Tạo voiced_mask dựa trên năng lượng tương đối và F0 hợp lệ.
    rms_p95 = float(np.percentile(rms, 95)) + 1e-8
    rms_rel = rms / rms_p95
    energy_threshold = max(float(np.percentile(rms_rel, 20)), 0.03)
    voiced_mask = (rms_rel > energy_threshold) & np.isfinite(f0) & (f0 > 0)

    f0_raw = f0.copy().astype(np.float32)
    f0_raw[~voiced_mask] = 0.0

    # Chỉ z-score F0 trên vùng voiced để tránh vùng vô thanh làm sai thống kê cao độ.
    f0_norm = np.zeros_like(f0_raw, dtype=np.float32)
    if voiced_mask.sum() > 1:
        f0_norm[voiced_mask] = safe_zscore_np(f0_raw[voiced_mask])

    energy_raw = rms.astype(np.float32)
    zcr_raw = zcr.astype(np.float32)
    # Chuẩn hóa energy để các file có thang đo gần nhau hơn.
    energy_norm = safe_zscore_np(energy_raw)
    # Chuẩn hóa ZCR/speaking activity để dùng ổn định ở notebook sau.
    zcr_norm = safe_zscore_np(zcr_raw)

    # Gom tất cả mảng đặc trưng để kiểm tra finite trước khi chuyển sang tensor.
    feature_arrays = {
        'mel': mel,
        'mel_abs': mel_abs,
        'mel_abs_db': mel_abs_db,
        'f0': f0_norm,
        'f0_raw': f0_raw,
        'energy': energy_norm,
        'energy_raw': energy_raw,
        'speaking_rate': zcr_norm,
        'speaking_activity': zcr_norm,
        'zcr': zcr_norm,
        'zcr_raw': zcr_raw,
        'voiced_mask': voiced_mask.astype(np.float32),
    }

    # Chặn NaN/Inf trước khi lưu feature; đây là điểm bảo vệ training khỏi lỗi dữ liệu.
    for name, x in feature_arrays.items():
        assert_finite_np(name, x)

    return {
        'mel': torch.from_numpy(mel.astype(np.float32)),
        'mel_abs': torch.from_numpy(mel_abs.astype(np.float32)),
        'mel_abs_db': torch.from_numpy(mel_abs_db.astype(np.float32)),
        'f0': torch.from_numpy(f0_norm.astype(np.float32)),
        'f0_raw': torch.from_numpy(f0_raw.astype(np.float32)),
        'energy': torch.from_numpy(energy_norm.astype(np.float32)),
        'energy_raw': torch.from_numpy(energy_raw.astype(np.float32)),
        'speaking_rate': torch.from_numpy(zcr_norm.astype(np.float32)),
        'speaking_activity': torch.from_numpy(zcr_norm.astype(np.float32)),
        'zcr': torch.from_numpy(zcr_norm.astype(np.float32)),
        'zcr_raw': torch.from_numpy(zcr_raw.astype(np.float32)),
        'voiced_mask': torch.from_numpy(voiced_mask.astype(np.float32)),
        'num_frames': int(T),
    }


# Hàm tạo một file feature .pt cho một processed audio, kèm metadata tối thiểu để training đọc lại.
def precompute_one(row: pd.Series) -> Dict[str, Any]:
    """Mỗi file .pt chứa mel/prosody/raw descriptors + sample_id/speaker/dataset/split/config."""
    # Đọc đường dẫn waveform .npy đã qua tiền xử lý.
    audio_path = Path(row['processed_path'])
    if not audio_path.exists():
        raise FileNotFoundError(f'Missing processed audio: {audio_path}')

    wav = np.load(audio_path).astype(np.float32)
    sr = int(row['sample_rate'])

    # Trích xuất feature từ waveform đã xử lý.
    feats = extract_features_from_waveform(wav, sr)
    out_path = FEATURE_OUT / row['feature_relpath']

    # Payload lưu cả feature tensor và metadata tối thiểu để training/evaluation truy vết nguồn gốc mẫu.
    payload = {
        **feats,
        'sample_id': row['sample_id'],
        'feature_id': row['feature_id'],
        'dataset': row['dataset'],
        'speaker_uid': row['speaker_uid'],
        'emotion': row['emotion'],
        'emotion_label': int(row['emotion_label']),
        'split': row['split'],
        'source_path': row['source_path'],
        'processed_path': row['processed_path'],
        'config': asdict(CFG),
    }
    # Lưu feature .pt theo kiểu atomic để tránh file bị hỏng khi Colab ngắt.
    atomic_torch_save(out_path, payload)

    return {
        'feature_path': str(out_path),
        'feature_status': 'ok',
        'num_frames': int(feats['num_frames']),
        'mel_shape': str(tuple(feats['mel'].shape)),
    }


# Hàm này chạy precompute_one cho toàn bộ processed metadata và lưu log thành công/thất bại.
def precompute_all(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Tạo feature_metadata_ok.csv, feature_metadata_failed.csv; đây là đầu vào chính cho các notebook training."""
    # Kiểm tra các cột bắt buộc trước khi precompute để lỗi metadata được phát hiện sớm.
    required_before = [
        'sample_id', 'feature_id', 'split', 'processed_path', 'feature_relpath',
        'speaker_uid', 'emotion', 'emotion_label', 'dataset'
    ]
    missing_before = [c for c in required_before if c not in df.columns]
    if missing_before:
        raise RuntimeError(f'precompute_all input missing columns: {missing_before}')

    records = []
    errors = []

    # Duyệt từng mẫu đã xử lý audio để tạo feature file.
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Precompute features'):
        base = row.to_dict()
        try:
            info = precompute_one(row)
            base.update(info)
            records.append(base)
        except Exception as e:
            base['feature_status'] = 'failed'
            base['feature_error'] = repr(e)
            errors.append(base)
            if CFG.fail_fast:
                raise

    ok = pd.DataFrame(records)
    err = pd.DataFrame(errors)

    # Lưu log feature OK để notebook sau đọc hoặc audit.
    ok.to_csv(OUT_ROOT / 'feature_metadata_ok.csv', index=False)
    # Lưu log feature failed để kiểm tra nguyên nhân lỗi.
    err.to_csv(OUT_ROOT / 'feature_metadata_failed.csv', index=False)

    print('Feature OK:', len(ok))
    print('Feature failed:', len(err))
    if len(err):
        print(err['feature_error'].value_counts().head(25))

    if len(ok) == 0:
        raise RuntimeError('No feature file generated successfully')

    return ok.reset_index(drop=True), err.reset_index(drop=True)

Kiểm tra cuối

In [ ]:
# Hàm bổ sung các trường ID số cuối cùng như speaker_label và dataset_id trước khi lưu all_metadata.csv.
def add_final_metadata_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Tạo mapping speaker/dataset để notebook training có thể dùng nhãn số nhất quán."""
    df = df.copy().reset_index(drop=True)

    # Tạo speaker_label dạng số từ speaker_uid để mô hình/adversary có thể sử dụng.
    speaker_uids = sorted(df['speaker_uid'].unique())
    speaker_to_id = {spk: i for i, spk in enumerate(speaker_uids)}
    df['speaker_label'] = df['speaker_uid'].map(speaker_to_id).astype(int)

    with open(OUT_ROOT / 'speaker_map.json', 'w', encoding='utf-8') as f:
        json.dump(speaker_to_id, f, indent=2, ensure_ascii=False)

    # Tạo dataset_id dạng số để audit hoặc kiểm tra domain effect.
    dataset_names = sorted(df['dataset'].unique())
    dataset_to_id = {d: i for i, d in enumerate(dataset_names)}
    df['dataset_id'] = df['dataset'].map(dataset_to_id).astype(int)

    with open(OUT_ROOT / 'dataset_map.json', 'w', encoding='utf-8') as f:
        json.dump(dataset_to_id, f, indent=2, ensure_ascii=False)

    # Các trường tương thích để notebook sau hoặc báo cáo dễ đọc hơn.
    df['emotion_name'] = df['emotion']
    df['filename'] = df['feature_id'] + '.wav'
    return df


# Hàm kiểm tra toàn bộ metadata và feature cuối cùng trước khi khóa dữ liệu cho huấn luyện.
def audit_final_metadata(df: pd.DataFrame):
    """Audit gồm: thiếu cột, trùng ID, thiếu file, LUFS lỗi, speaker leakage, trùng RAVDESS, shape feature và NaN/Inf."""
    print('=' * 80)
    print('FINAL AUDIT — KLTN SAFE DATA PIPELINE V10')
    print('=' * 80)

    # Nhóm 1: kiểm tra schema metadata cuối có đủ cột bắt buộc.
    required = [
        'sample_id', 'feature_id', 'dataset', 'dataset_id', 'speaker_uid', 'speaker_label',
        'emotion', 'emotion_label', 'split', 'processed_path', 'feature_path',
        'duration_final', 'lufs_final', 'num_frames'
    ]
    missing_cols = [c for c in required if c not in df.columns]
    if missing_cols:
        raise RuntimeError(f'Missing required metadata columns: {missing_cols}')

    # Nhóm 2: kiểm tra trùng ID/đường dẫn; trùng có thể gây leakage hoặc overwrite feature.
    checks = {
        'Duplicate sample_id': df['sample_id'].duplicated().sum(),
        'Duplicate feature_id': df['feature_id'].duplicated().sum(),
        'Duplicate processed_path': df['processed_path'].duplicated().sum(),
        'Duplicate feature_path': df['feature_path'].duplicated().sum(),
    }
    for k, v in checks.items():
        print(f'{k}: {v}')
        if v != 0:
            raise RuntimeError(f'{k} = {v}')

    # Nhóm 3: kiểm tra file processed audio còn tồn tại trên Drive.
    missing_audio = [p for p in df['processed_path'].tolist() if not Path(p).exists()]
    # Nhóm 3: kiểm tra file feature còn tồn tại trên Drive.
    missing_feat = [p for p in df['feature_path'].tolist() if not Path(p).exists()]
    print('Missing processed audio:', len(missing_audio))
    print('Missing feature files:', len(missing_feat))
    if missing_audio or missing_feat:
        raise RuntimeError('Missing processed audio or feature files')

    # Nhóm 4: kiểm tra LUFS cuối để loại file gần im lặng hoặc đo loudness lỗi.
    bad_lufs = df[(~np.isfinite(df['lufs_final'])) | (df['lufs_final'] < CFG.min_final_lufs_valid)]
    print('Bad LUFS final rows:', len(bad_lufs))
    if len(bad_lufs) > 0:
        bad_lufs.to_csv(AUDIT_OUT / 'bad_lufs_final_rows.csv', index=False)
        raise RuntimeError(f'Invalid lufs_final detected: {len(bad_lufs)} rows')

    # Nhóm 5: kiểm tra speaker-disjoint; đây là điều kiện quan trọng nhất để chống speaker leakage.
    for a, b in [('train', 'val'), ('train', 'test'), ('val', 'test')]:
        A = set(df[df['split'] == a]['speaker_uid'])
        B = set(df[df['split'] == b]['speaker_uid'])
        overlap = A & B
        print(f'Speaker overlap {a}/{b}:', len(overlap))
        if overlap:
            raise RuntimeError(f'Speaker leakage {a}/{b}: {len(overlap)}')

    # Nhóm 6: kiểm tra riêng RAVDESS vì dễ bị duplicate nếu folder có bản copy.
    rav = df[df['dataset'] == 'ravdess']
    if len(rav):
        dup_rav_utt = rav['utterance_id'].duplicated().sum()
        print('Duplicate RAVDESS utterance_id:', dup_rav_utt)
        if dup_rav_utt > 0:
            rav[rav['utterance_id'].duplicated(keep=False)].to_csv(
                AUDIT_OUT / 'duplicate_ravdess_utterance_id.csv', index=False
            )
            raise RuntimeError(f'Duplicate RAVDESS utterance_id detected: {dup_rav_utt}')

        rav_total = len(rav)
        print('RAVDESS final total:', rav_total)
        if rav_total > 1000:
            raise RuntimeError(f'RAVDESS final total seems too high: {rav_total}')

    # Nhóm 7: in các thống kê chính để đưa vào bảng Chương 4.
    print('\nSplit x Dataset')
    print(pd.crosstab(df['split'], df['dataset']))

    print('\nSplit x Emotion')
    print(pd.crosstab(df['split'], df['emotion']))

    print('\nDuration final')
    print(df['duration_final'].describe())

    print('\nLUFS final by dataset')
    print(df.groupby('dataset')['lufs_final'].describe())

    print('\nFeature frames')
    print(df['num_frames'].describe())

    # Nhóm 8: kiểm tra ngẫu nhiên payload feature .pt có đủ key cần cho training.
    required_feature_keys = [
        'mel', 'mel_abs', 'mel_abs_db',
        'f0', 'f0_raw', 'energy', 'energy_raw',
        'speaking_rate', 'speaking_activity', 'zcr', 'zcr_raw',
        'voiced_mask', 'num_frames', 'sample_id', 'feature_id',
        'dataset', 'speaker_uid', 'emotion', 'emotion_label', 'split', 'processed_path'
    ]

    sample_n = min(CFG.audit_random_feature_checks, len(df))
    # Lấy mẫu ngẫu nhiên một số feature để audit nhanh thay vì mở toàn bộ 9.894 file.
    sample_df = df.sample(sample_n, random_state=CFG.split_seed)

    for _, row in tqdm(sample_df.iterrows(), total=sample_n, desc='Audit feature payloads'):
        p = Path(row['feature_path'])
        # Mở feature .pt và kiểm tra shape/length/finite để bảo đảm training đọc được.
        obj = torch.load(p, weights_only=False, map_location='cpu')

        missing = [k for k in required_feature_keys if k not in obj]
        if missing:
            raise RuntimeError(f'Feature file missing keys {missing}: {p}')

        mel = obj['mel']
        if mel.ndim != 2 or mel.shape[0] != CFG.n_mels:
            raise RuntimeError(f'Bad mel shape {mel.shape}: {p}')

        # T là số frame theo thời gian; mọi đặc trưng 1D sẽ được căn về T.
        T = mel.shape[1]
        if int(obj['num_frames']) != T:
            raise RuntimeError(f'num_frames mismatch: {p}')

        for k in ['f0', 'f0_raw', 'energy', 'energy_raw', 'speaking_rate', 'zcr_raw', 'voiced_mask']:
            if len(obj[k]) != T:
                raise RuntimeError(f'{k} length mismatch in {p}')

        for k in ['mel', 'mel_abs', 'mel_abs_db', 'f0', 'f0_raw', 'energy', 'energy_raw', 'speaking_rate', 'zcr_raw']:
            x = obj[k]
            if not torch.isfinite(x).all():
                raise RuntimeError(f'Non-finite values in {k}: {p}')

        if row['sample_id'] != obj['sample_id']:
            raise RuntimeError(f'sample_id mismatch: {p}')

    # Lưu bảng split x dataset để có thể chèn hoặc đối chiếu khi viết Chương 4.
    pd.crosstab(df['split'], df['dataset']).to_csv(AUDIT_OUT / 'split_x_dataset.csv')
    pd.crosstab(df['split'], df['emotion']).to_csv(AUDIT_OUT / 'split_x_emotion.csv')
    df.groupby('dataset')['lufs_final'].describe().to_csv(AUDIT_OUT / 'lufs_final_by_dataset.csv')
    df['duration_final'].describe().to_csv(AUDIT_OUT / 'duration_final_describe.csv')

    # Xuất hình phân phối duration/LUFS để kiểm tra chất lượng pipeline trực quan.
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df, x='duration_final', hue='dataset', bins=40, element='step')
    plt.title('Duration final by dataset')
    plt.tight_layout()
    plt.savefig(AUDIT_OUT / 'duration_final_hist.png', dpi=160)
    plt.close()

    # Xuất hình phân phối duration/LUFS để kiểm tra chất lượng pipeline trực quan.
    plt.figure(figsize=(8, 5))
    sns.histplot(data=df, x='lufs_final', hue='dataset', bins=40, element='step')
    plt.title('LUFS final by dataset')
    plt.tight_layout()
    plt.savefig(AUDIT_OUT / 'lufs_final_hist.png', dpi=160)
    plt.close()

    print('\nAUDIT PASSED')
    print('Audit plots saved to:', AUDIT_OUT)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CFG = PreprocessConfig()

with open(OUT_ROOT / 'config.json', 'w', encoding='utf-8') as f:
    json.dump(asdict(CFG), f, indent=2, ensure_ascii=False)

print('Output root:', OUT_ROOT)
print(json.dumps(asdict(CFG), indent=2))

# parse metadata thô từ ba dataset.
raw_meta = parse_all_datasets()
raw_meta.to_csv(OUT_ROOT / 'raw_metadata.csv', index=False)

print('Raw metadata:', raw_meta.shape)
print(raw_meta.groupby(['dataset', 'emotion']).size())

# xử lý waveform và ghi metadata OK/failed.
processed_meta, processed_failed = process_all_audio(raw_meta)

# chia speaker-disjoint trước khi tạo feature để split được lưu vào feature payload.
split_meta = speaker_disjoint_split(processed_meta)
split_meta.to_csv(OUT_ROOT / 'split_metadata.csv', index=False)

# in các thống kê chính
print('\nSplit x Dataset')
print(pd.crosstab(split_meta['split'], split_meta['dataset']))
print('\nSplit x Emotion')
print(pd.crosstab(split_meta['split'], split_meta['emotion']))

# trích xuất và lưu feature .pt cho từng mẫu processed.
feature_meta, feature_failed = precompute_all(split_meta)

# thêm các trường ID số cuối cùng để training dùng được.
final_meta = add_final_metadata_fields(feature_meta)

front_cols = [
    'sample_id', 'feature_id', 'dataset', 'dataset_id', 'source_path', 'relative_path',
    'filename', 'utterance_id', 'speaker_local_id', 'speaker_uid', 'speaker_label',
    'emotion', 'emotion_name', 'emotion_label', 'raw_emotion', 'split',
    'processed_path', 'feature_path', 'processed_relpath', 'feature_relpath',
    'original_duration', 'duration_after_vad_or_fallback', 'duration_final',
    'sample_rate', 'lufs_before', 'lufs_after_norm', 'lufs_final', 'lufs_gain_db',
    'peak_final', 'vad_fallback_used', 'vad_reason', 'vad_trim_ratio',
    'num_frames', 'mel_shape'
]
front_cols = [c for c in front_cols if c in final_meta.columns]
other_cols = [c for c in final_meta.columns if c not in front_cols]
final_meta = final_meta[front_cols + other_cols]

final_meta.to_csv(OUT_ROOT / 'all_metadata.csv', index=False)

compact_cols = [
    'sample_id', 'feature_id', 'dataset', 'dataset_id', 'speaker_uid', 'speaker_label',
    'emotion', 'emotion_label', 'split', 'feature_path', 'processed_path',
    'duration_final', 'lufs_final', 'num_frames'
]
compact_cols = [c for c in compact_cols if c in final_meta.columns]
final_meta[compact_cols].to_csv(OUT_ROOT / 'metadata_compact.csv', index=False)

# audit cuối cùng; chỉ khi pass mới xem dữ liệu là bản khóa để huấn luyện.
audit_final_metadata(final_meta)

print('Saved final metadata:', OUT_ROOT / 'all_metadata.csv')
print('Saved compact metadata:', OUT_ROOT / 'metadata_compact.csv')
print('Final shape:', final_meta.shape)
print('MotionEmo V10 safe data pipeline v2 completed successfully.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output root: /content/drive/MyDrive/data/processed_v10_safe
{
  "sample_rate": 16000,
  "mono": true,
  "dtype": "float32",
  "target_lufs": -23.0,
  "peak_guard": 0.98,
  "min_lufs_valid": -60.0,
  "min_final_lufs_valid": -50.0,
  "max_gain_db": 35.0,
  "vad_frame_ms": 30.0,
  "vad_hop_ms": 10.0,
  "vad_db_margin": 12.0,
  "vad_keep_silence_ms": 200.0,
  "vad_min_voiced_ms": 60.0,
  "max_vad_trim_ratio": 0.55,
  "vad_fallback_to_untrimmed": true,
  "min_duration_sec": 1.2,
  "max_duration_sec": 8.0,
  "crop_long_strategy": "center",
  "n_mels": 80,
  "n_fft": 512,
  "win_length": 400,
  "hop_length": 160,
  "f_min": 50.0,
  "f_max": 8000.0,
  "f0_min": 50.0,
  "f0_max": 500.0,
  "top_db": 80.0,
  "use_iemocap_impro_only": true,
  "ravdess_speech_only": true,
  "ravdess_dedupe_by_stem": true,
  "train_ratio": 0.7,
  "val_ratio": 0.15,
  "test_ratio": 0.15,
  

Parse IEMOCAP:   0%|          | 0/10039 [00:00<?, ?it/s]

RAVDESS duplicate stem groups: 1440
Saved duplicate stem report: /content/drive/MyDrive/data/processed_v10_safe/audit/ravdess_duplicate_stems_before_selection.json
RAVDESS wav files found: 2880
RAVDESS selected unique stems: 1440


Parse RAVDESS:   0%|          | 0/1440 [00:00<?, ?it/s]

Parse CREMA-D:   0%|          | 0/7442 [00:00<?, ?it/s]

...

Processed OK: 9894
Failed: 92
error
ValueError('Too short before VAD: 1.140s')    5
ValueError('Too short before VAD: 1.100s')    5
ValueError('Too short before VAD: 1.190s')    4
ValueError('Too short before VAD: 1.020s')    4
ValueError('Too short before VAD: 1.130s')    4
ValueError('Too short before VAD: 1.120s')    3
ValueError('Too short before VAD: 1.000s')    3
ValueError('Too short before VAD: 1.160s')    3
ValueError('Too short before VAD: 1.040s')    3
ValueError('Too short before VAD: 1.060s')    3
ValueError('Too short before VAD: 1.090s')    3
ValueError('Too short before VAD: 1.110s')    3
ValueError('Too short before VAD: 1.106s')    2
ValueError('Too short before VAD: 0.940s')    2
ValueError('Too short before VAD: 1.197s')    2
ValueError('Too short before VAD: 1.080s')    2
ValueError('Too short before VAD: 1.150s')    2
ValueError('Too short before VAD: 1.070s')    2
ValueError('Too short before VAD: 0.764s')    1
ValueError('Too short before VAD: 1.109s')    1
Valu

Precompute features:   0%|          | 0/9894 [00:00<?, ?it/s]

Feature OK: 9894
Feature failed: 0
FINAL AUDIT — MOTIONEMO V10 SAFE DATA PIPELINE v2
Duplicate sample_id: 0
Duplicate feature_id: 0
Duplicate processed_path: 0
Duplicate feature_path: 0
Missing processed audio: 0
Missing feature files: 0
Bad LUFS final rows: 0
Speaker overlap train/val: 0
Speaker overlap train/test: 0
Speaker overlap val/test: 0
Duplicate RAVDESS utterance_id: 0
RAVDESS final total: 862

Split x Dataset
dataset  crema  iemocap  ravdess
split                           
test       947      560      144
train     4273     1731      574
val        950      571      144

Split x Emotion
emotion  anger  fear  happiness  neutral  sadness
split                                            
test       243   233        432      432      311
train     1191  1010       1560     1430     1387
val        315   228        378      383      361

Duration final
count    9894.000000
mean        2.946019
std         1.406817
min         1.200000
25%         2.130000
50%         2.530000
75

Audit feature payloads:   0%|          | 0/64 [00:00<?, ?it/s]


AUDIT PASSED
Audit plots saved to: /content/drive/MyDrive/data/processed_v10_safe/audit
Saved final metadata: /content/drive/MyDrive/data/processed_v10_safe/all_metadata.csv
Saved compact metadata: /content/drive/MyDrive/data/processed_v10_safe/metadata_compact.csv
Final shape: (9894, 56)
MotionEmo V10 safe data pipeline v2 completed successfully.
